# 2 Использование рага - эмбединги и ключи

## 1.Цель тетрадки

В первой тетрадке мы подготовили enriched chunks и построили локальный Qdrant index. Во второй тетрадке проверяем, насколько хорошо этот индекс отвечает на разные типы поисковых запросов.

Проверяем три уровня retrieval:

1. обычный dense search по embedding similarity;
2. metadata-search по структурированным полям `quote_author` и `interpretation_type`;
3. metadata-aware reranking, где dense search отвечает за смысл запроса, а metadata добавляет бонусы релевантным chunk.

Главная цель тетрадки — понять ограничения одного только semantic search и показать, зачем нам понадобились дополнительные поисковые сценарии для MCP tools.

In [1]:
from hw_rag_mcp.settings import get_settings

settings = get_settings()
settings.print_summary()

✓ Конфигурация загружена
  Env file:             /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/.env
  Repo root:            /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp
  LLM:                  GigaChat-2-MAX
  Embeddings:           EmbeddingsGigaR
  Qdrant path:          /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/data/vectorstore/qdrant
  Qdrant collection:    history_essay_chunks
  Langfuse base URL:    https://cloud.langfuse.com
  GigaChat credentials: заданы
  Langfuse credentials: заданы


# 2. Подключение к готовому Qdrant index

Перед поисковыми экспериментами подключаемся к уже созданному Qdrant index. Важно, что здесь мы не пересоздаём индекс, а открываем persistent-хранилище, подготовленное на этапе ingest.

Модель embeddings должна совпадать с моделью, которая использовалась при индексации. Если размерности embedding-векторов отличаются, Qdrant не сможет сравнивать query vector с сохранёнными vectors.

In [2]:
from langchain_gigachat import GigaChatEmbeddings

from hw_rag_mcp.vectorstore.qdrant_indexer import QdrantIndexer

In [3]:
embedding_model = GigaChatEmbeddings(
    credentials=settings.gigachat_credentials,
    scope=settings.gigachat_scope,
    model=settings.gigachat_embeddings_model,
    verify_ssl_certs=False,
)

print("✓ Модель эмбеддингов инициализирована")
print(f"  Model: {settings.gigachat_embeddings_model}")

indexer = QdrantIndexer(
    path=settings.qdrant_path,
    collection_name=settings.qdrant_collection,
    embedding_model=embedding_model,
)

print(f"Points count: {indexer.count()}")

✓ Модель эмбеддингов инициализирована
  Model: EmbeddingsGigaR
Points count: 255


# 3. Первые dense search
Сначала проверяем обычный semantic search. Это базовый retrieval-сценарий: пользователь задаёт естественный вопрос, мы строим embedding запроса и ищем ближайшие chunks в Qdrant.

Ожидаем, что dense search хорошо сработает для широких смысловых запросов: исторический период, событие, процесс, личность, реформа. Но он может быть слабее для запросов, где важны структурированные признаки: автор цитаты или тип авторской позиции.
## 3.1 Поиск темы по определенному периоду - ОК, работает

In [4]:
results = indexer.search(
    query="Найди темы про Киевскую Русь и объединение восточных славян",
    k=5,
)

for result in results:
    print("=" * 80)
    print(f"Rank: {result.rank}")
    print(f"Score: {result.score:.4f}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Period: {result.historical_period}")
    print()
    print(result.display_text)

Rank: 1
Score: 0.8738
Chunk ID: final_2015_topic_001
Document ID: final_2015
Source: final_2015.pdf
Period: Киевская Русь конца IX-X веков

Тема исторического эссе №1.
Документ: final_2015.
Источник: final_2015.pdf.
Год: 2015.
Этап олимпиады: final.
Классы: 10-11.

Автор цитаты: В.В. Мавродин.
Исторический период: Киевская Русь конца IX-X веков.
Ключевые слова: Киевская держава; конец IX-X веков; восточное славянство; объединение русских земель; единый государственный организм; рубежи Руси; общность языка и культуры.
Тип интерпретации: positive_assessment.
Позиция автора: Автор положительно оценивает Киевскую державу как силу, объединявшую восточных славян и защищавшую рубежи Руси.
Аннотация: Тема посвящена роли Киевской державы в объединении восточного славянства в конце IX-X веков.

Текст темы:
«Несмотря на примитивный характер объединения русских земель и племён, Киевская держава в лице её политических деятелей делала великое дело: объединяла восточное славянство в единый государств

### Вывод по 3.1

Dense search хорошо справился с обычным смысловым запросом. В выдаче появились темы, связанные с нужным историческим периодом и близкими историческими процессами.

Это подтверждает, что `embedding_text` работает как поисковое представление chunk: модель не ищет только точные совпадения слов, а находит смыслово близкие темы.

## 3.2 Поиск по определенному периоду + позиция автора - НЕ ОК, не очень работает

In [5]:
results = indexer.search(
    query="Существуют ли сочинения, в которых автор цитаты отрицательно высказывается о личности Александра Невского?",
    k=5,
)

for result in results:
    print("=" * 80)
    print(f"Rank: {result.rank}")
    print(f"Score: {result.score:.4f}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Period: {result.historical_period}")
    print()
    print(result.display_text)

Rank: 1
Score: 0.7682
Chunk ID: reg_2016_topic_002
Document ID: reg_2016
Source: reg_2016.pdf
Period: Русь XIII века

Тема исторического эссе №2.
Документ: reg_2016.
Источник: reg_2016.pdf.
Год: 2016.
Этап олимпиады: regional.
Классы: 10-11.

Автор цитаты: А.Е. Пресняков.
Исторический период: Русь XIII века.
Ключевые слова: нашествие Батыя; Владимирское княжество; Александр Невский; шведы; ливонские немцы; татары; оборона западных областей.
Тип интерпретации: mixed_assessment.
Позиция автора: Автор показывает Александра Невского как последнюю яркую фигуру Владимирского княжества, сочетавшую борьбу на Западе с осторожной покорностью Орде.
Аннотация: Тема посвящена положению Владимирского княжества после нашествия Батыя и деятельности Александра Невского.

Текст темы:
«После нашествия Батыя резко падает авторитет Владимирского княжества. Последней яркой вспышкой его энергии была деятельность Александра Невского – боевая в обороне западных областей от шведов и ливонских немцев, полная пол

### Вывод по 3.2

Здесь dense search работает хуже. Запрос содержит не только исторический объект, но и требование к авторской позиции: например, отрицательная, положительная, сравнительная или причинная интерпретация.

Embedding search может найти темы, близкие по историческому содержанию, но он не гарантирует совпадение по `interpretation_type` (поле из metadata). Поэтому для таких запросов одного dense search недостаточно: нужен дополнительный metadata-сигнал.

## 3.3 Поиск по автору - тоже НЕ ОК

In [6]:
results = indexer.search(
    query="Были ли темы, основанные на цитате Иосифа Сталина?",
    k=5,
)

for result in results:
    print("=" * 80)
    print(f"Rank: {result.rank}")
    print(f"Score: {result.score:.4f}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Period: {result.historical_period}")
    print()
    print(result.display_text)

Rank: 1
Score: 0.7161
Chunk ID: final_2023_topic_011
Document ID: final_2023
Source: final_2023.pdf
Period: СССР конца 1930-х годов

Тема исторического эссе №11.
Документ: final_2023.
Источник: final_2023.pdf.
Год: 2023.
Этап олимпиады: final.
Классы: 10-11.

Автор цитаты: Я.С. Драбкин.
Исторический период: СССР конца 1930-х годов.
Ключевые слова: советская идеология; интернационализм; русская державность; национализм; пакт Молотова-Риббентропа; 1939 год; Сталин; Гитлер.
Тип интерпретации: negative_assessment.
Позиция автора: Автор критически трактует пакт 1939 года как вершину замещения интернационализма националистическим сталинизмом.
Аннотация: Тема посвящена изменениям советской идеологии в конце 1930-х годов.

Текст темы:
«В советской официальной идеологии и пропаганде демонстративный интернационализм все более уступал место возрождавшимся традициям русской державности и национализма… Без сомнения, «зияющей вершиной» (если воспользоваться метафорой Александра Зиновьева) подмены ин

### Вывод по 3.3

Запрос по автору цитаты также показывает ограничение dense search. Если пользователь спрашивает темы у конкретного автора, например Сталина, Карамзина или Вернадского, это лучше проверять не через embedding similarity, а через структурированное поле `quote_author`.

Автор цитаты — это metadata, а не смысловая тема. Поэтому поиск по автору должен быть отдельным use case, а не побочным эффектом semantic search.

# 4. Выводы по dense search: зачем нужны дополнительные виды поиска

На этом шаге мы проверили, что локальный Qdrant-индекс успешно открывается из второй тетрадки и возвращает top-k результаты по semantic similarity.

Dense search хорошо работает для широких смысловых запросов. Например, если пользователь спрашивает про Киевскую Русь, объединение восточных славян или русские земли, semantic retrieval может найти релевантные темы даже без точного совпадения формулировок.

Однако в нашем корпусе есть поля, которые по своей природе являются не столько семантическим текстом, сколько структурированными metadata:

- `quote_author` — автор цитаты;
- `interpretation_type` — тип авторской позиции;
- `year`, `stage`, `source`, `document_id`, `chunk_id`. Год тоже можно было бы использовать, но не будем усложнять.

Для таких полей dense search работает нестабильно. Например, запрос по автору цитаты может вернуть похожие исторические темы, но не обязательно темы нужного автора. Аналогично запрос вида “отрицательная оценка Александра Невского” состоит из двух разных частей: исторический объект лучше искать семантически, а отрицательную оценку лучше проверять по полю `interpretation_type`.

Поэтому одного dense search недостаточно. В дальнейшей логике добавляются два дополнительных вида поиска:

1. поиск по автору цитаты;
2. поиск по типу авторской интерпретации.

Эти виды поиска не заменяют dense retrieval, а дополняют его. В финальной системе их можно объединить через простую fusion-стратегию: dense search находит исторически близкие темы, metadata-search уточняет автора или тип оценки, а итоговый список ранжируется по совокупности сигналов.



# 5. Два новых вида поиска

После первых экспериментов видно, что dense search полезен, но не закрывает все типы запросов.

Для нашего корпуса особенно важны два структурированных признака:

- `quote_author` — автор цитаты;
- `interpretation_type` — тип авторской позиции.

Эти поля были подготовлены на этапе enrichment и сохранены в payload Qdrant. Поэтому их можно использовать не как текст для embeddings, а как metadata для точного поиска и reranking.

После проверки dense search добавим два metadata-based поиска.

Первый поиск — по автору цитаты. Он нужен для запросов вида:

- “Были ли темы на основе цитаты Сталина?”
- “Какие темы встречались у В.О. Ключевского?”
- “Найди темы по Вернадскому.”

Второй поиск — по типу авторской интерпретации. Он нужен для запросов, где пользователь явно или неявно спрашивает о характере оценки:

- положительная оценка;
- отрицательная оценка;
- сравнительная трактовка;
- ревизионистская позиция;
- причинно-следственное объяснение.

Эти два поиска не используют embeddings. Они работают по структурированным metadata, которые были подготовлены на этапе нормализации JSON.

## 5.1 Поиск по автору - все ОК

Во всех metadata-search методах значение `k=-1` используется как специальный режим “вернуть все совпадения”. Это удобно для reranking: dense search формирует ограниченный список кандидатов, а metadata-search отдаёт полное множество chunk_id, которые подходят под условие.

In [7]:
author_results = indexer.search_by_quote_author(
    author="Сталин",
)

for result in author_results:
    print("=" * 80)
    print(f"Rank: {result.rank}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Quote author: {result.quote_author}")
    print(f"Period: {result.historical_period}")
    print(f"Type: {result.interpretation_type}")
    print()
    print(result.display_text)

Rank: 1
Chunk ID: final_2018_topic_017
Document ID: final_2018
Source: final_2018.pdf
Quote author: И.В. Сталин
Period: Сталинский период и Великая Отечественная война
Type: positive_assessment

Тема исторического эссе №17.
Документ: final_2018.
Источник: final_2018.pdf.
Год: 2018.
Этап олимпиады: final.
Классы: 10-11.

Автор цитаты: И.В. Сталин.
Исторический период: Сталинский период и Великая Отечественная война.
Ключевые слова: И.В. Сталин; советский строй; мобилизация народа; военное время; экономический подъём; культурный подъём; Великая Отечественная война; мирное строительство.
Тип интерпретации: positive_assessment.
Позиция автора: Автор апологетически оценивает советский строй как лучшую форму мобилизации общества в мирное и военное время.
Аннотация: Тема посвящена официальной оценке советской системы после войны.

Текст темы:
«Уроки войны говорят о том, что Советский строй оказался не только лучшей формой организации экономического и культурного подъема страны в годы мирного 

### Вывод по 5.1

Поиск по `quote_author` работает как точный metadata-search. Он не пытается определить смысловую близость, а возвращает chunks, где автор цитаты совпадает с заданным значением.

Это полезно для запросов вида “найди темы у Сталина” или “покажи цитаты Вернадского”. В будущей MCP-интеграции такой сценарий удобно вынести в отдельный tool.

## 5.2 Поиск по авторской оценке - работает, но не достаточно

In [8]:
type_results = indexer.search_by_interpretation_type(
    interpretation_type="negative_assessment",
    k=5,
)

for result in type_results:
    print("=" * 80)
    print(f"Rank: {result.rank}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Quote author: {result.quote_author}")
    print(f"Period: {result.historical_period}")
    print(f"Type: {result.interpretation_type}")
    print()
    print(result.display_text)

Rank: 1
Chunk ID: final_2015_topic_002
Document ID: final_2015
Source: final_2015.pdf
Quote author: Дж. Феннел
Period: Русь XIII века
Type: negative_assessment

Тема исторического эссе №2.
Документ: final_2015.
Источник: final_2015.pdf.
Год: 2015.
Этап олимпиады: final.
Классы: 10-11.

Автор цитаты: Дж. Феннел.
Исторический период: Русь XIII века.
Ключевые слова: Русь XIII века; татарское иго; княжеские роды; политическая раздробленность; консерватизм князей; устаревший порядок; ордынская зависимость.
Тип интерпретации: negative_assessment.
Позиция автора: Автор критически объясняет слабость Руси XIII века внутренним консерватизмом и неспособностью князей к изменениям.
Аннотация: Тема посвящена причинам слабости Руси XIII века в условиях ордынского нашествия и раздробленности.

Текст темы:
«Слабость Руси XIII столетия была вызвана не столько внешними факторами или так называемым татарским игом, сколько преступным консерватизмом, органически присущим правившим княжеским родам, их нежела

### Вывод по 5.2

Поиск по `interpretation_type` возвращает все темы с заданным типом авторской позиции. Сам по себе такой поиск слишком широкий: например, `negative_assessment` может относиться к разным периодам и разным историческим деятелям.

Поэтому `interpretation_type` лучше использовать не как самостоятельную финальную выдачу, а как metadata-сигнал для усиления dense search. Dense search найдёт смысловые candidates, а `interpretation_type` поможет поднять выше те chunks, которые совпали по типу оценки.

# 6. Metadata-aware reranking

В этой тетрадке мы не используем полноценный Reciprocal Rank Fusion, потому что дополнительные поиски по metadata не являются ранжированными списками.

Dense search возвращает ранжированный список кандидатов со score. А поиск по `quote_author` или `interpretation_type` работает как фильтр: chunk либо удовлетворяет условию, либо нет.

Поэтому используется более простая стратегия metadata-aware reranking:

1. сначала получаем top-N кандидатов через dense search;
2. отдельно получаем множества chunk_id, подходящие под metadata-условия;
3. каждому dense-кандидату добавляется бонус, если он присутствует в одном из metadata-множеств;
4. результаты сортируются по `final_score`.

Так semantic search отвечает за историческое содержание запроса, а metadata-бонусы усиливают результаты, которые также совпали по автору цитаты или типу авторской позиции.

Для metadata-фильтров в примерах используется `k=-1`, потому что здесь нам важно получить не top-k, а полное множество подходящих chunk_id. Эти результаты затем используются не как самостоятельная выдача, а как источник бонусов для dense-кандидатов.

In [9]:
def apply_metadata_boosts(
    dense_results,
    bonus_result_lists: dict[str, list],
    bonus_weights: dict[str, float],
    top_k: int = 5,
):
    bonus_chunk_ids_by_name = {
        name: {
            result.chunk_id
            for result in results
            if result.chunk_id is not None
        }
        for name, results in bonus_result_lists.items()
    }

    boosted_results = []

    for result in dense_results:
        final_score = float(result.score)
        applied_bonuses = []

        for bonus_name, bonus_chunk_ids in bonus_chunk_ids_by_name.items():
            if result.chunk_id in bonus_chunk_ids:
                bonus_value = bonus_weights.get(bonus_name, 0.0)
                final_score += bonus_value

                applied_bonuses.append(
                    {
                        "name": bonus_name,
                        "bonus": bonus_value,
                    }
                )

        boosted_results.append(
            {
                "result": result,
                "dense_rank": result.rank,
                "dense_score": float(result.score),
                "final_score": final_score,
                "applied_bonuses": applied_bonuses,
            }
        )

    boosted_results.sort(
        key=lambda item: item["final_score"],
        reverse=True,
    )

    return [
        {
            **item,
            "final_rank": final_rank,
        }
        for final_rank, item in enumerate(boosted_results[:top_k], start=1)
    ]

## 6.1 Отрицательная оценка А. Невского - успех

In [10]:
dense_results = indexer.search(
    query="Существуют ли сочинения, в которых автор цитаты отрицательно высказывается о личности Александра Невского?",
    k=20,
)

negative_results = indexer.search_by_interpretation_type(
    interpretation_type="negative_assessment",
    k=100,
)

boosted_results = apply_metadata_boosts(
    dense_results=dense_results,
    bonus_result_lists={
        "negative_assessment": negative_results,
    },
    bonus_weights={
        "negative_assessment": 0.1,
    },
    top_k=5,
)

for item in boosted_results:
    result = item["result"]

    print("=" * 80)
    print(f"Final rank: {item['final_rank']}")
    print(f"Final score: {item['final_score']:.4f}")
    print(f"Dense rank: {item['dense_rank']}")
    print(f"Dense score: {item['dense_score']:.4f}")
    print(f"Applied bonuses: {item['applied_bonuses']}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Quote author: {result.quote_author}")
    print(f"Period: {result.historical_period}")
    print(f"Type: {result.interpretation_type}")
    print()
    print(result.display_text)

Final rank: 1
Final score: 0.8456
Dense rank: 4
Dense score: 0.7456
Applied bonuses: [{'name': 'negative_assessment', 'bonus': 0.1}]
Chunk ID: reg_2018_topic_002
Document ID: reg_2018
Source: reg_2018.pdf
Quote author: Дж. Феннел
Period: Русь второй половины XIII века
Type: negative_assessment

Тема исторического эссе №2.
Документ: reg_2018.
Источник: reg_2018.pdf.
Год: 2018.
Этап олимпиады: regional.
Классы: 10-11.

Автор цитаты: Дж. Феннел.
Исторический период: Русь второй половины XIII века.
Ключевые слова: Александр Невский; вторая половина XIII века; татарское давление; татарский гнёт; ордынская зависимость; татарские войска; преемники Александра Невского; политика русских князей.
Тип интерпретации: negative_assessment.
Позиция автора: Автор критически связывает укрепление татарского гнёта с политикой Александра Невского и его преемников.
Аннотация: Тема посвящена оценке политики русских князей в условиях ордынской зависимости.

Текст темы:
«Вторая половина XIII в. была эпохой пос

Metadata-aware reranking улучшил результат по сравнению с обычным dense search. Он вывел тему Дж. Феннел на 1-е место (вместо 4). На второе место выдвинулась тема Н.И. Костомаров. Он отрицательно оценивает Дмитрия Донского. На мой экспертный взгляд, это даже хорошо: отрицательно о Донском примерно = отрицательно о Невском при написании сочинения, поэтому я буду рад получить такой ответ (тема Н.И. Костомарова) на мой вопрос о Невском.

## 6.2 Демонстрация поддержка нескольких фильтров

In [11]:
dense_results = indexer.search(
    query="Сравнение развития Руси и Западной Европы в период Раннего Средневековья",
    k=30,
)

author_results = indexer.search_by_quote_author(
    author="Вернадский",
    k=100,
)

comparative_results = indexer.search_by_interpretation_type(
    interpretation_type="comparative",
    k=100,
)

boosted_results = apply_metadata_boosts(
    dense_results=dense_results,
    bonus_result_lists={
        "quote_author": author_results,
        "comparative_assessment": comparative_results,
    },
    bonus_weights={
        "quote_author": 0.20,
        "comparative_assessment": 0.15,
    },
    top_k=5,
)

for item in boosted_results:
    result = item["result"]

    print("=" * 80)
    print(f"Final rank: {item['final_rank']}")
    print(f"Final score: {item['final_score']:.4f}")
    print(f"Dense rank: {item['dense_rank']}")
    print(f"Dense score: {item['dense_score']:.4f}")
    print(f"Applied bonuses: {item['applied_bonuses']}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Document ID: {result.document_id}")
    print(f"Source: {result.source}")
    print(f"Quote author: {result.quote_author}")
    print(f"Period: {result.historical_period}")
    print(f"Type: {result.interpretation_type}")
    print()
    print(result.display_text)

Final rank: 1
Final score: 1.1815
Dense rank: 3
Dense score: 0.8315
Applied bonuses: [{'name': 'quote_author', 'bonus': 0.2}, {'name': 'comparative_assessment', 'bonus': 0.15}]
Chunk ID: final_2023_topic_001
Document ID: final_2023
Source: final_2023.pdf
Quote author: Г.В. Вернадский
Period: Киевская Русь
Type: comparative

Тема исторического эссе №1.
Документ: final_2023.
Источник: final_2023.pdf.
Год: 2023.
Этап олимпиады: final.
Классы: 10-11.

Автор цитаты: Г.В. Вернадский.
Исторический период: Киевская Русь.
Ключевые слова: Киевская Русь; монетарная экономика; натуральное хозяйство; город; феодальное поместье; Западная Европа; социальная эволюция.
Тип интерпретации: comparative.
Позиция автора: Автор сравнивает социально-экономическое развитие Руси и Западной Европы, подчёркивая особую роль города на Руси.
Аннотация: Тема посвящена экономической и социальной специфике Киевской Руси.

Текст темы:
«В Киевской Руси… монетарная экономика превалировала над натуральным хозяйством. И, в 

In [12]:
indexer.close()
print("✓ Qdrant client closed")

✓ Qdrant client closed


Этот пример показывает, что metadata-aware reranking может учитывать несколько metadata-сигналов одновременно.

Dense search отвечает за смысловую часть запроса, а дополнительные фильтры усиливают chunks, которые совпадают по структурированным признакам. Например, можно одновременно учитывать автора цитаты и тип интерпретации.

Такой подход удобен для будущего MCP tool: агент может выбрать более точный поисковый сценарий, если пользователь явно просит автора цитаты или определённый тип авторской позиции.

## 7. Итоги

Во второй тетрадке мы проверили качество поиска по подготовленному Qdrant index и определили, какие retrieval-сценарии реально нужны нашему агенту.

Сначала был протестирован обычный dense search. Он хорошо подходит для широких смысловых запросов: например, для поиска тем про Киевскую Русь, реформы Петра I, Смутное время, советскую индустриализацию или развитие русской государственности. Такой поиск полезен, когда пользователь формулирует запрос естественным языком и не использует точные слова из исходной цитаты.

При этом эксперименты показали ограничение одного только dense search. В нашем корпусе есть важные структурированные признаки:

- `quote_author` — автор цитаты;
- `interpretation_type` — тип авторской позиции.

Эти metadata были добавлены на этапе enrichment не случайно. Для RAG-системы это нормальная и полезная практика: часть информации лучше хранить не только в тексте для embeddings, но и в структурированном виде. Это позволяет делать более точные поисковые сценарии, например “найди темы у Сталина” или “найди тему про Александра Невского с отрицательной оценкой автора”.

В тетрадке мы также проверили более сложную идею — metadata-aware reranking. В этой схеме dense search сначала находит смысловых кандидатов, а metadata-сигналы добавляют бонусы результатам, которые совпали по автору или типу интерпретации. Такой подход действительно может улучшать ранжирование в смешанных запросах.

Однако для финальной версии проекта мы не будем усложнять систему полноценным reranking/fusion-слоем. Это учебное задание про RAG + MCP, и нам важнее сделать простую, понятную и воспроизводимую архитектуру. Поэтому metadata-aware reranking в этой тетрадке остаётся как исследовательская демонстрация: мы показали, что так можно делать и почему metadata в RAG полезны.

В финальный MCP-сервер будут вынесены более простые и прозрачные retrieval-сценарии:

| Сценарий | Назначение |
|---|---|
| `semantic_search` | обычный смысловой поиск по теме |
| `search_by_quote_author` | поиск по автору цитаты |
| `search_by_query_and_interpretation_type` | смысловой поиск с учётом типа авторской позиции |

Такой вариант проще объяснить, легче отлаживать и удобнее демонстрировать через агента. Агент получает несколько отдельных MCP tools и сам выбирает подходящий инструмент по пользовательскому запросу.

Итог второй тетрадки: мы убедились, что индекс работает, metadata возвращаются и могут использоваться в поиске, а для финальной агентной демонстрации достаточно трёх понятных MCP-инструментов вместо сложного ранжирующего pipeline.